# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeroTechno/Machine-Learning/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Content Action Playbook Mapping:
Model probability scores alone are not enough for an editorial team—they need clear, actionable decisions. We convert the Random Forest probability outputs into a ranked action queue with explicit reason codes:   
Action 1 (OPTIMIZE_TITLE): Triggered when avg_position <= 10 and ctr < 0.02. Reason Code: HIGH_RANK_LOW_CTR.
Focuses on improving meta-titles and snippets for high-visibility pages that fail to capture clicks.

Action 2 (REFRESH_CONTENT): Triggered when days_since_last_update > 180 and impressions_90d > 100.
Reason Code: STALE_HIGH_TRAFFIC. Focuses on updating outdated information on proven traffic drivers.

Action 3 (MONITOR_AND_MAINTAIN): Default for stable or lower-volume pages. Reason Code: STABLE_PERFORMANCE.

In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# 1. Load Data
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url).fillna(0)

# 2. Target Proxy Definition
df["target"] = ((df["impressions_90d"] >= 100) & (df["ctr"] >= 0.05)).astype(int)

features = ["avg_position", "impressions_90d", "days_since_last_update", "ctr", "word_count"]
X = df[features]
y = df["target"]

# 3. Train Random Forest Model
rf_model = RandomForestClassifier(max_depth=5, random_state=42)
rf_model.fit(X, y)
df["model_probability"] = rf_model.predict_proba(X)[:, 1]

# 4. Map Actions & Reason Codes
def assign_playbook_action(row):
    if row["avg_position"] <= 10 and row["ctr"] < 0.02:
        return "OPTIMIZE_TITLE", "HIGH_RANK_LOW_CTR"
    elif row["days_since_last_update"] > 180 and row["impressions_90d"] > 100:
        return "REFRESH_CONTENT", "STALE_HIGH_TRAFFIC"
    else:
        return "MONITOR_AND_MAINTAIN", "STABLE_PERFORMANCE"

df[["recommended_action", "reason_code"]] = df.apply(assign_playbook_action, axis=1, result_type="expand")

# 5. Build Ranked Queue
ranked_playbook = df.sort_values(by="model_probability", ascending=False).reset_index(drop=True)

print("=== RANKED ACTION PLAYBOOK SUMMARY ===")
print(ranked_playbook["recommended_action"].value_counts())

=== RANKED ACTION PLAYBOOK SUMMARY ===
recommended_action
MONITOR_AND_MAINTAIN    24118
OPTIMIZE_TITLE           5853
REFRESH_CONTENT            29
Name: count, dtype: int64


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*
Intended Purpose & Operational Boundaries:   Intended Purpose: Serves strictly as a decision-support tool for editorial and SEO teams to prioritize monthly content re-optimization budgets (targeting the top 50 recommendations per cycle).   
Operational Limits:   
Non-Causal: The model ranks potential priority based on historical search signals; it does not guarantee SERP rank or traffic movement.
Static Aggregate Snapshot: Operates on 90-day aggregate Search Console metrics and cannot account for real-time daily search engine volatility.

No Cost/ROI Calculation: The model does not factor in the author/editor hourly costs required for complex content rewrites.

In [7]:
# Profile the score distribution across action types to establish operational limits
action_limits_summary = ranked_playbook.groupby("recommended_action").agg(
    count=("model_probability", "count"),
    avg_prob=("model_probability", "mean"),
    avg_ctr=("ctr", "mean"),
    avg_impressions=("impressions_90d", "mean")
).reset_index()

print("=== ACTION TYPE PROFILES & LIMITS ===")
print(action_limits_summary.to_string(index=False))

=== ACTION TYPE PROFILES & LIMITS ===
  recommended_action  count  avg_prob  avg_ctr  avg_impressions
MONITOR_AND_MAINTAIN  24118  0.621043 0.634065      6373.633966
      OPTIMIZE_TITLE   5853  0.009390 0.000036       357.356228
     REFRESH_CONTENT     29  0.732131 1.014483      6899.275862


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human Review Protocol & The No-Go List:
Every recommendation generated by the model must undergo human editor validation before any live website modifications are made.   The No-Go List (Strictly Prohibited for Automated Execution):   
Premature Pages (days_since_last_update < 14): Brand-new pages exhibit erratic indexing CTRs while search engine algorithms evaluate them.
Legal & Compliance Pages: Privacy policy, terms of service, or legal disclaimers must never be auto-flagged or altered by AI rules.
Expired Seasonal Campaigns: Holiday or event-specific landing pages that show staleness due to natural off-season traffic drops.
Direct CMS Auto-Publishing: Model outputs must never trigger automatic live website updates without manual human sign-off.


In [8]:
# Flag No-Go cases in the ranked queue
ranked_playbook["no_go_flag"] = np.where(
    ranked_playbook["days_since_last_update"] < 14,
    "PREMATURE_PAGE_REVIEW_REQUIRED",
    "APPROVED_FOR_HUMAN_REVIEW"
)

no_go_count = (ranked_playbook["no_go_flag"] != "APPROVED_FOR_HUMAN_REVIEW").sum()
print(f"Total top queue items flagged for No-Go / Special Review: {no_go_count}")
print(ranked_playbook[["recommended_action", "reason_code", "no_go_flag"]].head(5))

Total top queue items flagged for No-Go / Special Review: 3428
     recommended_action         reason_code                 no_go_flag
0  MONITOR_AND_MAINTAIN  STABLE_PERFORMANCE  APPROVED_FOR_HUMAN_REVIEW
1  MONITOR_AND_MAINTAIN  STABLE_PERFORMANCE  APPROVED_FOR_HUMAN_REVIEW
2  MONITOR_AND_MAINTAIN  STABLE_PERFORMANCE  APPROVED_FOR_HUMAN_REVIEW
3  MONITOR_AND_MAINTAIN  STABLE_PERFORMANCE  APPROVED_FOR_HUMAN_REVIEW
4  MONITOR_AND_MAINTAIN  STABLE_PERFORMANCE  APPROVED_FOR_HUMAN_REVIEW


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

To prevent model decay and keep recommendations trustworthy over time:   
Weekly Precision@50 Monitoring: Evaluate top-50 queue predictions against newly logged Search Console performance data.
Retrain Trigger 1 (Performance Drift): If Precision@50 on rolling 30-day evaluation windows drops below 80.0%.
Retrain Trigger 2 (Search Ecosystem Shift): Triggered immediately following major Google Core Algorithm updates or domain structural migrations.
Retrain Trigger 3 (Data Schema Drift): Triggered if underlying data warehouse column definitions or tracking parameters change.

In [9]:
# Calculate current baseline performance for monitoring logs
top_50_precision = ranked_playbook.head(50)["target"].mean()

monitoring_log = {
    "current_top_50_precision": float(top_50_precision),
    "retrain_precision_threshold": 0.80,
    "status": "HEALTHY" if top_50_precision >= 0.80 else "RETRAIN_REQUIRED"
}

print("=== MODEL MONITORING STATUS ===")
print(monitoring_log)

=== MODEL MONITORING STATUS ===
{'current_top_50_precision': 1.0, 'retrain_precision_threshold': 0.8, 'status': 'HEALTHY'}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Artifact Exports for Deployed Research Paper:
This notebook exports the finalized queue CSV, performance receipt JSON, and distribution plot figure to work/outputs/ and work/figures/ for direct reuse in our research paper.

In [10]:
import os
import json
import matplotlib.pyplot as plt

# Ensure output directories exist
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1. Export Ranked Queue CSV
queue_export_path = "work/outputs/action_playbook_queue.csv"
ranked_playbook.to_csv(queue_export_path, index=False)
print(f"Exported ranked playbook queue to: {queue_export_path}")

# 2. Export Metrics JSON Receipt
metrics_receipt = {
    "model_name": "RandomForestClassifier",
    "total_pages_evaluated": len(ranked_playbook),
    "top_50_precision": float(top_50_precision),
    "actions_breakdown": ranked_playbook["recommended_action"].value_counts().to_dict()
}

json_export_path = "work/outputs/playbook_metrics.json"
with open(json_export_path, "w") as f:
    json.dump(metrics_receipt, f, indent=4)
print(f"Exported metrics receipt JSON to: {json_export_path}")

# 3. Export Action Distribution Plot Figure
plt.figure(figsize=(8, 4))
ranked_playbook["recommended_action"].value_counts().plot(kind="bar", color="#2563eb")
plt.title("Action Playbook Recommendation Distribution")
plt.xlabel("Recommended Action")
plt.ylabel("Number of Pages")
plt.xticks(rotation=15)
plt.tight_layout()

figure_export_path = "work/figures/action_distribution.png"
plt.savefig(figure_export_path)
plt.close()
print(f"Exported figure plot to: {figure_export_path}")

Exported ranked playbook queue to: work/outputs/action_playbook_queue.csv
Exported metrics receipt JSON to: work/outputs/playbook_metrics.json
Exported figure plot to: work/figures/action_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.